# AAPL Stock Price Predictive Analytics

This notebook explores Apple (AAPL) historical stock data and builds a simple machine-learning model to predict the next day's closing price.

**Note:** This is an educational project, not financial advice.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 1. Load the dataset

In [ ]:
# The notebook is inside notebooks/, while the dataset is inside data/.
data_path = Path('../data/AAPL_stock_data (1).csv')

df = pd.read_csv(data_path)
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nMissing values:')
print(df.isna().sum())

## 2. Clean and inspect the data

In [ ]:
# Standardize column names
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

# Detect the date and closing-price columns
date_candidates = [c for c in df.columns if 'date' in c]
close_candidates = [c for c in df.columns if c in ['close', 'closing_price', 'adj_close', 'adjusted_close'] or 'close' in c]

date_col = date_candidates[0] if date_candidates else df.columns[0]
close_col = close_candidates[0] if close_candidates else 'close'

df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
df[close_col] = pd.to_numeric(df[close_col], errors='coerce')
df = df.dropna(subset=[date_col, close_col]).sort_values(date_col).reset_index(drop=True)

print('Date column:', date_col)
print('Close column:', close_col)
df.head()

## 3. Visualize AAPL closing prices

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df[date_col], df[close_col])
plt.title('AAPL Historical Closing Price')
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Create predictive features

In [ ]:
# Use previous trading days' closing prices to predict the next closing price.
model_df = df[[date_col, close_col]].copy()
model_df['lag_1'] = model_df[close_col].shift(1)
model_df['lag_2'] = model_df[close_col].shift(2)
model_df['lag_3'] = model_df[close_col].shift(3)
model_df['lag_5'] = model_df[close_col].shift(5)
model_df['rolling_mean_5'] = model_df[close_col].rolling(5).mean()
model_df['rolling_mean_10'] = model_df[close_col].rolling(10).mean()
model_df = model_df.dropna().reset_index(drop=True)

features = ['lag_1', 'lag_2', 'lag_3', 'lag_5', 'rolling_mean_5', 'rolling_mean_10']
X = model_df[features]
y = model_df[close_col]

model_df.head()

## 5. Train the model

In [ ]:
# Time-series data should be split chronologically rather than randomly.
split_index = int(len(model_df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## 6. Evaluate the model

In [ ]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'MAE:  {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R²:   {r2:.4f}')

## 7. Compare actual vs predicted prices

In [ ]:
results = pd.DataFrame({
    'Date': model_df.loc[model_df.index[split_index:], date_col].values,
    'Actual': y_test.values,
    'Predicted': predictions
})

plt.figure(figsize=(12, 5))
plt.plot(results['Date'], results['Actual'], label='Actual')
plt.plot(results['Date'], results['Predicted'], label='Predicted')
plt.title('AAPL Actual vs Predicted Closing Price')
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

results.tail(10)

## 8. Predict the next closing price

The prediction below uses the most recent available lag and rolling features. It should be treated as a model output for learning purposes, not as a trading recommendation.

In [ ]:
latest_features = model_df[features].iloc[[-1]]
next_prediction = model.predict(latest_features)[0]
print(f'Predicted next closing price: {next_prediction:.2f}')

## Conclusion

- Historical AAPL prices were loaded and cleaned.
- Lag and rolling-average features were created.
- A Random Forest regression model was trained using a chronological train/test split.
- MAE, RMSE and R² were calculated to evaluate performance.
- Actual and predicted prices were visualized.

**Important:** Stock prices are affected by many factors that are not included in this simple model. Predictions are uncertain and should not be interpreted as financial advice.